# Data Description

Author: RUEI-HSIANG, CHANG（張睿庠；B12303030；AI協助）

Date first created: 2026-11-03

## Background

議員性別從717筆有值增至1502筆來源有值；785筆新增中784筆採2026次級來源，1筆僅收2022官方值並標明當屆衝突。完整率不是官方核實率。

## Unit of observation / Sample period

council_sex_current：2026登記名冊每候選人一列，1502人；candidate_fields_current為candidate_key＋field（含81名首長）；master每候選人一列；background每議員一列；observations每版本候選表一列；conflicts每來源差異一列。資料截至2026-11-03。

## Known issues

張光儀2022官方男與2026維基女仍有衝突，exclude_from_unqualified_2026_analysis=1，不裁定當屆性別。陳泱瑚同號次二手來源姓名及性別不同，保留舊官方值並列異常。4筆維基版本差異另列。次級來源可能互相抄錄；歷史同名同選區一致不等同2026官方資格／性別驗證。11筆姓名差異僅為來源對照，不改名冊。

## Definition of each variable

`council_sex_current`：candidate_key原登記鍵；candidate_name原名；county_city縣市；electoral_district選區；sex來源明載值；source_tier來源級別；source_year資料年份；source_url來源網址；source_file保存檔；source_locator列／原文定位；source_name來源寫法；match_method比對方式；identity_review_source拼字對照檔；official_2022_same_name_district_agreement同名同選區歷史一致之檔案及列（非自動身份認證）；previous_value更新前值；review_note差異紀錄；exclude_from_unqualified_2026_analysis當屆未決衝突旗標；checked_date核對日。

`observations`：county_city、district_number、electoral_district_source、ballot_number、source_name、sex、party_source、source_url、source_file、source_locator、source_version、row_text分別為來源縣市、選區編號／原文、號次、姓名、性別、政黨、網址、檔案、定位、版本、原始四欄文字。

`conflicts`：candidate_key、candidate_name、issue、source_a、value_a、source_b、value_b、resolution保存兩方來源及處理／未決狀態。

既有三表欄位定義延續[round9文件](municipal_roles_round9_20261019_documentation.ipynb)。background新增sex_focus_source_tier、sex_focus_review_note、sex_focus_current_conflict，定義同來源級別、差異紀錄、當屆未決旗標；sex_additional_source_year明列歷史／當屆，不全當作2026。verification JSON保存各輸出rows、columns、sha256及涵蓋計數。

## Data path / Source code / Role / Validation / Downstream

下方固定列出7個輸出檔案與雜湊。來源為[處理Notebook](../code/sex_focus_20261103.ipynb)及其可讀Python模組；輸入快照固定，原資料不刪。角色為蒐集／來源檢查表，尚非最終分析樣本。未重跑下游分析；使用當屆性別前須處理旗標與次級來源限制。

In [1]:
from pathlib import Path
import json, hashlib, sys
import pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'data').is_dir())
sys.path.insert(0,str(ROOT/'data/code'))
def sha(path): return hashlib.sha256((ROOT/path).read_bytes()).hexdigest()
def read(path): return pd.read_csv(ROOT/path,dtype=str,keep_default_na=False)
PINS={'data/processed/candidate_fields_current.csv': '74ef1f3a77d24f73a5f3682682b18c97ca55843e7879ada7c1cc86b57dd0730c', 'data/processed/candidate_master_current.csv': 'f4806b229118b4f67383e4ed081144bffc0b81404600ffce646df6017c23eea4', 'data/processed/council_background_20261019.csv': '32aebc40e4579333fa6f4d9476a763e1974537f6349f82f49396a88f7c58b33b', 'data/processed/council_sex_current.csv': '21518248cbedff13985bd7775e705b1dbee59abc343cc950a30da7f5cc68b0f8', 'data/processed/council_sex_source_observations_20261103.csv': '44b6cab74fceaa3801e30f9e964c30a48ba2625ea5a08e138db7964d745e1f0a', 'data/processed/council_sex_conflicts_20261103.csv': 'd193d3adbd8c9458827899e24d790b9a7476e1abf794f79edf35649e7f81c223', 'data/processed/council_sex_verification_20261103.json': 'b0008ccd3ed10bd69452495e8240593efabf0007a8f09340197a0d18366a5a68'}
for p,h in PINS.items(): assert sha(p)==h,p
sex=read('data/processed/council_sex_current.csv')
assert len(sex)==1502 and sex.candidate_key.is_unique
assert sex.sex.isin(['男','女']).all()
assert sex.previous_value.eq('').sum()==785
assert sex.exclude_from_unqualified_2026_analysis.eq('1').sum()==1
print('Saved output hashes verified:',len(PINS))

Saved output hashes verified: 7


# Summary

## Sample

In [2]:
display(sex.head(5))

,candidate_key,candidate_name,county_city,electoral_district,sex,source_tier,source_year,source_url,source_file,source_locator,source_name,match_method,identity_review_source,official_2022_same_name_district_agreement,previous_value,review_note,exclude_from_unqualified_2026_analysis,checked_date
0,2026|直轄市議員|臺北市第1選舉區|謝閔弘,謝閔弘,臺北市,臺北市第1選舉區,男,secondary_2026_explicit_sex,2026,https://zh.wikipedia.org/zh-tw/2026年中華民國直轄市議員及...,data/raw/sex_focus_20261103/wiki_1.source,table 1; row 10,謝閔弘,same_county_district_unique_normalized_chinese...,,,,,0,2026-11-03
1,2026|直轄市議員|臺北市第1選舉區|李文,李文,臺北市,臺北市第1選舉區,女,secondary_2026_explicit_sex,2026,https://zh.wikipedia.org/zh-tw/2026年中華民國直轄市議員及...,data/raw/sex_focus_20261103/wiki_1.source,table 1; row 16,李文,same_county_district_unique_normalized_chinese...,,,,,0,2026-11-03
2,2026|直轄市議員|臺北市第1選舉區|陳彥廷,陳彥廷,臺北市,臺北市第1選舉區,男,secondary_2026_explicit_sex,2026,https://zh.wikipedia.org/zh-tw/2026年中華民國直轄市議員及...,data/raw/sex_focus_20261103/wiki_1.source,table 1; row 11,陳彥廷,same_county_district_unique_normalized_chinese...,,,,,0,2026-11-03
3,2026|直轄市議員|臺北市第1選舉區|鍾佩玲,鍾佩玲,臺北市,臺北市第1選舉區,女,prior_record_retained,see_prior_evidence,https://db.cec.gov.tw/ | data/raw/cec_all_offi...,data/processed/snapshots/pre_sex_focus_2026110...,2026|直轄市議員|臺北市第1選舉區|鍾佩玲; field=sex,鍾佩玲,prior_evidence_preserved,,data/raw/cec_all_offices/107_elcand.csv:10,女,,0,2026-11-03
4,2026|直轄市議員|臺北市第1選舉區|紀建漢,紀建漢,臺北市,臺北市第1選舉區,男,secondary_2026_explicit_sex,2026,https://zh.wikipedia.org/zh-tw/2026年中華民國直轄市議員及...,data/raw/sex_focus_20261103/wiki_1.source,table 1; row 7,紀建漢,same_county_district_unique_normalized_chinese...,,,,,0,2026-11-03


## Summary Stats / Assertions

In [3]:
display(sex.groupby(['source_tier','sex']).size().rename('count').reset_index())
display(sex.groupby('county_city').agg(candidates=('candidate_key','size'),populated=('sex','count')))
for p in PINS:
 if p.endswith('.csv'):
  d=read(p)
  if p.endswith('candidate_fields_current.csv'): assert not d.duplicated(['candidate_key','field']).any()
  elif 'observations' not in p and 'conflicts' not in p: assert d.candidate_key.is_unique
print('Reopened schemas, candidate keys and populated values checked.')

,source_tier,sex,count
0,media_2026_registration_explicit_sex,女,1
1,media_2026_registration_explicit_sex,男,2
2,official_2022_historical_current_conflict,男,1
3,prior_record_retained,女,285
4,prior_record_retained,男,432
5,secondary_2026_explicit_sex,女,252
6,secondary_2026_explicit_sex,男,529


,candidates,populated
county_city,,
南投縣,61,61
嘉義市,37,37
嘉義縣,53,53
基隆市,60,60
宜蘭縣,59,59
屏東縣,87,87
彰化縣,86,86
新北市,112,112
新竹市,56,56


Reopened schemas, candidate keys and populated values checked.
